In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers==4.36.0 peft==0.7.0 accelerate==0.25.0 evaluate datasets bitsandbytes

In [ ]:
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import load_from_disk
import torch
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# === CONFIG ===
EPOCHS = 3
LEARNING_RATE = 3e-5
BATCH_SIZE = 8
MAX_LENGTH = 384
DOC_STRIDE = 128

# Paths (adjust based on your Kaggle Datasets)
STAGE1_CHECKPOINT = "/kaggle/input/xlm-roberta-stage1-checkpoint/stage1_best"  # From Stage 1
DATASET_PATH = "/kaggle/input/viquad-normalized/viquad_normalized"  # Your ViQuAD dataset
OUTPUT_DIR = "/kaggle/working/stage2_output"
CHECKPOINT_DIR = "/kaggle/working/stage2_checkpoints"
FINAL_MODEL_DIR = "/kaggle/working/stage2_best"

## Load Stage 1 Checkpoint

In [ ]:
print(f"Loading Stage 1 checkpoint from {STAGE1_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(STAGE1_CHECKPOINT)
model = AutoModelForQuestionAnswering.from_pretrained(STAGE1_CHECKPOINT)

print("Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

## Load ViQuAD Dataset

In [ ]:
print(f"Loading ViQuAD from {DATASET_PATH}")
viquad = load_from_disk(DATASET_PATH)

print(f"Train size: {len(viquad['train'])}")
print(f"Validation size: {len(viquad['validation'])}")
print(f"Test size: {len(viquad['test'])}")

## Prepare Data

In [ ]:
def prepare_train_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue
        
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        token_start_index = 0
        while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        token_start_index -= 1
        
        token_end_index = len(offsets) - 1
        while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        token_end_index += 1
        
        sequence_ids = tokenized.sequence_ids(i)
        context_start = sequence_ids.index(1) if 1 in sequence_ids else 0
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1) if 1 in sequence_ids else len(sequence_ids)
        
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        elif token_start_index < context_start or token_end_index > context_end:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            tokenized["start_positions"].append(token_start_index)
            tokenized["end_positions"].append(token_end_index)
    
    return tokenized

print("Tokenizing datasets...")
train_dataset = viquad['train'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['train'].column_names,
    desc="Tokenizing train"
)

val_dataset = viquad['validation'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['validation'].column_names,
    desc="Tokenizing validation"
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

## Training

In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=500,  # Warm restart from EN checkpoint
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("="*60)
print("Starting Stage 2 Training (VI Fine-tune)")
print("="*60)

trainer.train()

## Save Final Model

In [ ]:
print(f"Saving final model to {FINAL_MODEL_DIR}")
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("="*60)
print("Stage 2 completed!")
print(f"Final model saved to: {FINAL_MODEL_DIR}")
print("="*60)
print("\nNext steps:")
print("1. Download /kaggle/working/stage2_best/ folder")
print("2. Use this model for inference or evaluation")
print("3. Upload to HuggingFace Hub (optional)")

In [ ]:
# Check output files
!ls -lh /kaggle/working/stage2_best/

## Quick Evaluation (Optional)

In [ ]:
# Test on a few examples
test_examples = viquad['validation'].select(range(5))

for i, example in enumerate(test_examples):
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    print(f"Question: {example['question']}")
    print(f"Context: {example['context'][:200]}...")
    print(f"Ground Truth: {example['answers']['text'][0]}")
    
    # Predict
    inputs = tokenizer(
        example['question'],
        example['context'],
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    prediction = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    
    print(f"Prediction: {prediction}")